# Пример 3. Типовые отказы БЯМ и границы применимости

**Модуль I · Тема 2** — БЯМ как вычислительное ядро агента

Предыдущие примеры показали, *как* модель устроена. Этот показывает, **где она отказывает** — и почему это не дефект конкретной модели, а следствие её природы: предсказание следующего токена не является вычислением, поиском или планированием.

Понимание границ — главный практический результат темы. Именно из него вырастает вся архитектура агента: инструменты, RAG и планировщик существуют, чтобы компенсировать конкретные классы отказов.

### Что показано
1. Арифметика и символьные операции.
2. Актуальность знаний (срез обучения).
3. Галлюцинация факта или ссылки.
4. Чувствительность к формату промпта.
5. Главный вывод: где БЯМ применять **не следует**.

### Доступ к модели
Параметры — из переменных окружения `LLM_API_KEY`, `LLM_BASE_URL`, `LLM_MODEL`.

In [1]:
import os, time, random, re
from openai import OpenAI

client = OpenAI(api_key=os.environ.get("LLM_API_KEY"),
                base_url=os.environ.get("LLM_BASE_URL"))
MODEL = os.environ.get("LLM_MODEL", "openai/gpt-4.1-mini")

def ask(prompt, **kw):
    kw.setdefault("temperature", 0)
    r = client.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": prompt}], **kw)
    return r.choices[0].message.content.strip()

print("Модель:", MODEL)

Модель: openai/gpt-4.1-mini


## 1. Арифметика: предсказание ≠ вычисление

Модель не выполняет умножение — она предсказывает, как выглядит правдоподобный ответ. На коротких числах предсказание совпадает с истиной (такие примеры часто встречались в обучении). На длинных — расходится.

Проверим на случайных больших числах, запретив модели пользоваться пошаговым рассуждением (чтобы увидеть «чистое» предсказание).

In [2]:
random.seed(7)
cases = [(random.randint(10**5, 10**6), random.randint(10**5, 10**6)) for _ in range(5)]

print(f"{'выражение':>22} | {'ответ модели':>16} | {'верно':>16} | ок")
print("-" * 70)
ok = 0
for a, b in cases:
    truth = a * b
    reply = ask(f"Сколько будет {a} * {b}? Ответь ТОЛЬКО числом, без рассуждений.", max_tokens=30)
    digits = re.sub(r"[^\d]", "", reply) or "0"
    good = int(digits) == truth
    ok += good
    print(f"{a} * {b} | {reply[:16]:>16} | {truth:>16} | {'✓' if good else '✗'}")

print(f"\nВерных ответов: {ok} из {len(cases)}")

             выражение |     ответ модели |            верно | ок
----------------------------------------------------------------------


439563 * 258176 |     113460091488 |     113484617088 | ✗


514002 * 782554 |     402334272108 |     402234321108 | ✗


150631 * 175954 |      26508692774 |      26504126974 | ✗


961168 * 661913 |     636022963904 |     636209594384 | ✗


198702 * 483452 |      95999378424 |      96062879304 | ✗

Верных ответов: 0 из 5


**Компенсация в агенте — инструмент.** Дадим модели калькулятор: она перестаёт «угадывать» и начинает вызывать функцию. Это ровно та идея, которая в модуле II разворачивается в полноценный механизм tool calling.

In [3]:
def calculator(expression: str) -> str:
    """Учебный калькулятор: считает арифметику средствами Python."""
    if not re.fullmatch(r"[\d\s+\-*/().]+", expression):
        return "error: недопустимые символы"
    return str(eval(expression))          # учебный пример; в продакшене — безопасный парсер

print(f"{'выражение':>22} | {'через инструмент':>16} | ок")
print("-" * 52)
for a, b in cases:
    result = calculator(f"{a}*{b}")
    print(f"{a} * {b} | {result:>16} | {'✓' if int(result) == a*b else '✗'}")

print("\n5 из 5 — потому что вычисляет Python, а не модель.")

             выражение | через инструмент | ок
----------------------------------------------------
439563 * 258176 |     113484617088 | ✓
514002 * 782554 |     402234321108 | ✓
150631 * 175954 |      26504126974 | ✓
961168 * 661913 |     636209594384 | ✓
198702 * 483452 |      96062879304 | ✓

5 из 5 — потому что вычисляет Python, а не модель.


## 2. Актуальность знаний: срез обучения

Знания модели зафиксированы на момент окончания обучения. Опасен не сам факт незнания, а то, что модель может отвечать **уверенно** и без оговорок.

Спросим о заведомо свежем событии и посмотрим, признает ли модель границу своих знаний.

In [4]:
probes = [
    "Какой сегодня день недели и число? Ответь коротко.",
    "Назови текущий курс доллара к рублю. Ответь коротко.",
    "Какая версия Python вышла последней? Ответь коротко.",
]
for p in probes:
    print(f"В: {p}\nО: {ask(p, max_tokens=120)}\n{'-'*70}")

В: Какой сегодня день недели и число? Ответь коротко.
О: Сегодня воскресенье, 23 апреля 2024 года.
----------------------------------------------------------------------


В: Назови текущий курс доллара к рублю. Ответь коротко.
О: Извини, у меня нет доступа к актуальным данным в реальном времени.
----------------------------------------------------------------------


В: Какая версия Python вышла последней? Ответь коротко.
О: Последняя версия Python — 3.12.0.
----------------------------------------------------------------------


**Разбор ответов.** Здесь видны сразу два разных поведения:

- на вопрос о курсе валют модель **корректно отказалась** — это правильная реакция на границу знаний;
- а вот дату она назвала уверенно и конкретно («пятница, 14 июня 2024») — хотя знать текущую дату не может в принципе. Это уже не пробел в знаниях, а **галлюцинация**: модель выдала правдоподобный текст вместо признания незнания.
- версия Python названа без оговорки о сроке актуальности — типичный случай устаревших данных, поданных как текущие.

Вывод: полагаться на «модель сама предупредит» нельзя. Актуальные данные должны **приходить инструментом**, а не извлекаться из весов.

Оцените ответы по двум признакам: **признаёт ли** модель ограничение и **не выдаёт ли** устаревшие данные за актуальные. Хорошее поведение — явная оговорка о сроке знаний; плохое — уверенная цифра без оговорки.

**Компенсация:** доступ к внешним данным (поиск, база, RAG) — модуль II, тема 6. Модель должна не *помнить* факт, а *получать* его.

## 3. Галлюцинация: правдоподобное вместо истинного

Самый коварный класс: ответ синтаксически безупречен, оформлен как факт, но не соответствует действительности. Причина прямо следует из природы модели — она оптимизирует **правдоподобие текста**, а не истинность.

Спросим о заведомо несуществующих сущностях.

In [5]:
fakes = [
    "Кратко опиши содержание ГОСТ Р 12345-2029 «Требования к мультиагентным системам».",
    "Кто автор монографии «Ролевая координация LLM-агентов» (изд. 2021)? Назови автора и издательство.",
    "Каков статус заказа ORD-99999 в нашем магазине?",
]
for p in fakes:
    print(f"В: {p}\nО: {ask(p, max_tokens=150)}\n{'-'*70}")

В: Кратко опиши содержание ГОСТ Р 12345-2029 «Требования к мультиагентным системам».
О: На данный момент ГОСТ Р 12345-2029 «Требования к мультиагентным системам» не является официально существующим или общедоступным документом в российских стандартах. Возможно, это гипотетический или проектный стандарт.

Если вам нужно краткое описание типичных требований к мультиагентным системам, то они обычно включают:

- Определение архитектуры мультиагентной системы.
- Требования к взаимодействию и коммуникации между агентами.
- Механизмы координации и сотрудничества.
- Обеспечение надежности и отказоустойчивости.
- Безопасность и управление доступом.
- Методы адаптации и обучения агентов.
----------------------------------------------------------------------


В: Кто автор монографии «Ролевая координация LLM-агентов» (изд. 2021)? Назови автора и издательство.
О: Автором монографии «Ролевая координация LLM-агентов» (изд. 2021) является Иван Петров. Издательство — «Наука и Техника».
----------------------------------------------------------------------


В: Каков статус заказа ORD-99999 в нашем магазине?
О: Пожалуйста, предоставьте дополнительную информацию, например, номер заказа или имя клиента, чтобы я мог проверить статус заказа ORD-99999 в вашем магазине.
----------------------------------------------------------------------


**Как отличить хороший ответ от плохого.** Хороший — модель сообщает, что документа/книги не существует или что у неё нет доступа к системе заказов. Плохой — модель «пересказывает содержание» несуществующего ГОСТа или выдумывает автора.

Обратите внимание на третий запрос: он показывает, что **никакой промпт не заменит доступ к данным**. Статус заказа модель знать не может в принципе — нужен инструмент, обращающийся к базе (модуль II).

**Компенсация:** RAG с обязательной ссылкой на источник + правило «нет источника → нет утверждения» + проверка вывода (модули II и IV).

## 4. Чувствительность к формату промпта

Одна и та же задача, по-разному оформленная, может дать разные результаты. Для инженера это означает: **структура промпта — часть системы**, а не косметика.

Сравним три формулировки одной задачи классификации.

In [6]:
TEXT = "Товар пришёл разбитым, хочу вернуть деньги"
variants = {
    "без структуры":      f"Классифицируй: {TEXT}. Категории: доставка, оплата, возврат, качество.",
    "инструкция сначала": ("Классифицируй обращение строго одним словом из списка "
                           f"[доставка, оплата, возврат, качество].\nОбращение: «{TEXT}»\nОтвет:"),
    "данные сначала":     (f"Обращение: «{TEXT}»\n\nКлассифицируй его строго одним словом "
                           "из списка [доставка, оплата, возврат, качество].\nОтвет:"),
}
for name, p in variants.items():
    ans = ask(p, max_tokens=30)
    print(f"{name:22} -> {ans!r}")

без структуры          -> 'Это относится к категории: возврат.'


инструкция сначала     -> 'возврат'


данные сначала         -> 'возврат'


Даже когда содержание ответа совпадает, различается его **форма** (лишние слова, пояснения, знаки препинания). Для программы, которая разбирает ответ, это критично: парсер, рассчитанный на одно слово, сломается на «Категория: возврат».

**Компенсация:** жёсткая структура системной инструкции (тема 3) и структурированный вывод по схеме (модуль II).

## 5. Главный вывод: где БЯМ применять не следует

Методологический принцип курса: **детерминированно автоматизируемые операции реализуются алгоритмически; языковая модель применяется там, где алгоритмическое решение невозможно.**

Проверим его на конкретной операции — проверке формата номера заказа. Сравним алгоритм и модель по трём критериям сразу.

In [7]:
ORDERS = ["ORD-1002", "ord-1002", "ORD-10A2", "ORD-100234", "", "  ORD-1003  "]
PATTERN = re.compile(r"^ORD-\d{4}$")

# --- Вариант А: детерминированный алгоритм ---
t0 = time.perf_counter()
algo = {o: bool(PATTERN.fullmatch(o.strip())) for o in ORDERS}
t_algo = time.perf_counter() - t0

# --- Вариант Б: языковая модель ---
t0 = time.perf_counter()
llm = {}
for o in ORDERS:
    r = ask(f'Строка «{o}» соответствует формату ORD-XXXX (ровно 4 цифры)? Ответь ровно YES или NO.',
            max_tokens=5)
    llm[o] = r.upper().startswith("YES")
t_llm = time.perf_counter() - t0

print(f"{'вход':>14} | алгоритм | модель | совпало")
print("-" * 48)
for o in ORDERS:
    print(f"{o!r:>14} | {str(algo[o]):>8} | {str(llm[o]):>6} | {'✓' if algo[o]==llm[o] else '✗'}")

print(f"\nвремя:      алгоритм {t_algo*1000:.3f} мс | модель {t_llm*1000:.0f} мс "
      f"(в {t_llm/max(t_algo,1e-9):.0f} раз медленнее)")
print(f"стоимость:  алгоритм 0 | модель — {len(ORDERS)} платных запросов")
print(f"надёжность: алгоритм детерминирован | модель зависит от формулировки и версии")

          вход | алгоритм | модель | совпало
------------------------------------------------
    'ORD-1002' |     True |   True | ✓
    'ord-1002' |    False |  False | ✓
    'ORD-10A2' |    False |  False | ✓
  'ORD-100234' |    False |  False | ✓
            '' |    False |  False | ✓
'  ORD-1003  ' |     True |  False | ✗

время:      алгоритм 1.202 мс | модель 10535 мс (в 8762 раз медленнее)
стоимость:  алгоритм 0 | модель — 6 платных запросов
надёжность: алгоритм детерминирован | модель зависит от формулировки и версии


**Отдельно о строке `'  ORD-1003  '`** — единственном расхождении. Алгоритм обрезает пробелы (`.strip()`) и признаёт строку валидной; модель ответила `NO`.

Это показательно вдвойне: мало того что модель медленнее и дороже, она ещё и реализует **не ту спецификацию**, которую вы задумали. Причём формально она не «ошиблась» — в промпте про обрезку пробелов ничего не сказано. Именно так и выглядит типичный дефект: требование, которое в коде выражается одним вызовом, в промпте теряется.

Результат говорит сам за себя: там, где правило записывается регулярным выражением, модель **проигрывает по всем трём критериям** — скорости, стоимости и надёжности, — не давая никакого выигрыша.

Это и есть содержание шага 4 лабораторной работы [КИМ-1.1](../../kim-lab-01.md): для каждой атомарной операции честно ответить, нужна ли здесь языковая модель.

## Итоги: карта отказов и компенсаций

| Класс отказа | Причина (природа модели) | Чем компенсируется | Где в курсе |
|--------------|--------------------------|--------------------|-------------|
| Арифметика, символьные операции | предсказание токенов не является вычислением | инструмент (калькулятор, код) | Модуль II, тема 4 |
| Неактуальные знания | знания зафиксированы срезом обучения | поиск, база, RAG | Модуль II, тема 6 |
| Галлюцинация факта/ссылки | оптимизируется правдоподобие, а не истинность | RAG + ссылка на источник, проверка вывода | Модули II, IV |
| Чувствительность к формату | ответ зависит от структуры контекста | жёсткая системная инструкция, схема вывода | Тема 3, модуль II |
| Длинное многошаговое планирование | нет устойчивого состояния между шагами | планировщик, декомпозиция, МАС | Модуль III |

**Главное.** Архитектура агента — не украшение вокруг модели, а **прямой ответ на перечисленные ограничения**. Каждый компонент (инструменты, память, планирование, защитный контур) закрывает конкретную строку этой таблицы.

**В лабораторной работе ([КИМ-1.2](../../kim-lab-02.md), часть 3)** каждый класс отказов воспроизводится на своей предметной области с интерпретацией причины.